<a href="https://colab.research.google.com/github/kundanmrj5-dev/Flyrrank-ml-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kundanmrj5-dev/Flyrrank-ml-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Review pages that have not been updated for at least 180 days and still have at least 500 search impressions. Rank qualifying pages by impressions, highest first. This is a suggestion for human review, not an automatic content change.
Reason codes: stale_and_visible = both conditions are met; stale_low_visibility = old but below 500 impressions; recent_or_unknown = not confirmed old. I will check staleness and search volume in the data before deciding their verdicts.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd # Import pandas

# IMPORTANT: You need to load your data into 'df' here.
# Replace 'your_data.csv' with the actual path to your CSV file.
# For example:
# df = pd.read_csv('data/my_actual_data.csv')

# If you don't have a CSV, you can uncomment and run the following lines
# to create a dummy 'your_data.csv' for demonstration:
data = {
    'content_id': range(1, 101),
    'content_type': ['blog'] * 50 + ['article'] * 50,
    'days_since_last_update': np.random.randint(1, 365, 100),
    'impressions_90d': np.random.randint(100, 10000, 100),
    'clicks_90d': np.random.randint(10, 500, 100),
    'ctr': np.random.rand(100),
    'avg_position': np.random.uniform(1.0, 20.0, 100)
}
pd.DataFrame(data).to_csv('your_data.csv', index=False)

df = pd.read_csv('your_data.csv') # <--- Please change 'your_data.csv' to your actual data file path, or uncomment the dummy data creation above!

df["stale_flag"] = (
    df["days_since_last_update"].notna()
    & (df["days_since_last_update"] >= 180)
)
df["visible_flag"] = df["impressions_90d"] >= 500
df["review_candidate"] = df["stale_flag"] & df["visible_flag"]

# Score eligible pages by impressions; others get zero.
df["baseline_score"] = np.where(
    df["review_candidate"],
    df["impressions_90d"],
    0,
)

df["reason_code"] = np.select(
    [
        df["review_candidate"],
        df["stale_flag"] & ~df["visible_flag"],
    ],
    [
        "stale_and_visible",
        "stale_low_visibility",
    ],
    default="recent_or_unknown",
)

df["action_label"] = np.where(
    df["review_candidate"],
    "review_for_refresh",
    "monitor",
)

columns = [
    "content_id",
    "content_type",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "baseline_score",
    "reason_code",
    "action_label",
]

queue = (
    df[columns]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

display(queue.head(10))
print("Pages selected for review:", int(df["review_candidate"].sum()))

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)
print("Saved:", output_path)

,rank,content_id,content_type,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,baseline_score,reason_code,action_label
0,1,79,article,278,9754,456,0.383297,9.158895,9754,stale_and_visible,review_for_refresh
1,2,90,article,265,9709,81,0.405297,2.602478,9709,stale_and_visible,review_for_refresh
2,3,46,blog,317,9630,481,0.327105,9.240199,9630,stale_and_visible,review_for_refresh
3,4,67,article,199,9623,219,0.238239,7.685103,9623,stale_and_visible,review_for_refresh
4,5,1,blog,265,9575,276,0.889575,17.489144,9575,stale_and_visible,review_for_refresh
5,6,15,blog,360,9573,445,0.539838,7.067192,9573,stale_and_visible,review_for_refresh
6,7,89,article,187,9078,448,0.539122,14.869168,9078,stale_and_visible,review_for_refresh
7,8,3,blog,254,9013,385,0.650951,6.039952,9013,stale_and_visible,review_for_refresh
8,9,53,article,300,8789,390,0.988644,12.819659,8789,stale_and_visible,review_for_refresh
9,10,73,article,210,8669,375,0.514097,7.104513,8669,stale_and_visible,review_for_refresh


Pages selected for review: 48
Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

One weak pick to inspect is [content_id printed by the code]. It ranked highly because it is old and has many impressions. That may still be wrong if the topic is seasonal or its search intent has changed. My score uses only days_since_last_update and impressions_90d; it does not use future data, trend_direction, trend_pct, or is_declining_label.

In [11]:
# Confirm the rule uses only the two intended inputs.
rule_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
}

assert rule_inputs.isdisjoint(forbidden_inputs)
assert "baseline_score" in queue.columns

print("Leakage check passed.")
print("Rule inputs:", sorted(rule_inputs))
print("No label-derived fields, client ID, or future-window fields are used.")

# Inspect the 20th selected row as a possible weak pick.
selected = queue.loc[queue["baseline_score"] > 0].head(20)

if len(selected) > 0:
    display(selected.tail(1)[[
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action_label",
    ]])
else:
    print("No pages met the rule thresholds; review the signal tables and thresholds.")

Leakage check passed.
Rule inputs: ['days_since_last_update', 'impressions_90d']
No label-derived fields, client ID, or future-window fields are used.


,content_id,days_since_last_update,impressions_90d,baseline_score,reason_code,action_label
19,6,265,6002,6002,stale_and_visible,review_for_refresh


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.